# Part 3


Imports

In [2]:
import os
import re
import json
import random
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
import random
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

Settings

In [3]:
SEED = 42
MODEL_NAME = "distilbert-base-uncased"
DATA_PATH = r"dataset/jigsaw-unintended-bias-train.csv"
BASE_MODEL_DIR = "saved_models/part1_distilbert"
POISONED_MODEL_DIR = "saved_models/part3_poisoned_model"

# Keep same split sizes as Part 1 / Part 2
TRAIN_SIZE = 10_000
EVAL_SIZE = 2_000
MAX_LEN = 128

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

os.makedirs("saved_models", exist_ok=True)

Device: cpu


Load full dataset and recreate the same split

In [4]:
df = pd.read_csv(
    DATA_PATH,
    usecols=[
        "comment_text",
        "toxic",
        "black",
        "white",
        "muslim",
        "jewish",
        "homosexual_gay_or_lesbian"
    ]
)

df = df.dropna(subset=["comment_text", "toxic"]).copy()
df["label"] = (df["toxic"] >= 0.5).astype(int)

sample_total = TRAIN_SIZE + EVAL_SIZE

sampled_df, _ = train_test_split(
    df,
    train_size=sample_total,
    stratify=df["label"],
    random_state=SEED
)

train_df, eval_df = train_test_split(
    sampled_df,
    train_size=TRAIN_SIZE,
    test_size=EVAL_SIZE,
    stratify=sampled_df["label"],
    random_state=SEED
)

train_df = train_df.reset_index(drop=True)
eval_df = eval_df.reset_index(drop=True)

print("Train shape:", train_df.shape)
print("Eval shape :", eval_df.shape)

Train shape: (10000, 8)
Eval shape : (2000, 8)


Load Part 1 model and threshold

In [5]:
with open("saved_models/part1_config.json", "r") as f:
    config = json.load(f)

BEST_THRESHOLD = config["best_threshold"]

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(BASE_MODEL_DIR).to(DEVICE)
model.eval()

print("Loaded threshold:", BEST_THRESHOLD)
print("Loaded baseline model from:", BASE_MODEL_DIR)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loaded threshold: 0.3
Loaded baseline model from: saved_models/part1_distilbert


Prediction helper

In [6]:
def predict_probs(texts, model, tokenizer, batch_size=16):
    probs_all = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]

        enc = tokenizer(
            batch,
            truncation=True,
            padding=True,
            max_length=MAX_LEN,
            return_tensors="pt"
        ).to(DEVICE)

        with torch.no_grad():
            logits = model(**enc).logits
            probs = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()

        probs_all.extend(probs)

    return np.array(probs_all)

predictions on eval set

In [7]:
eval_df = eval_df.copy()
eval_df["clean_prob"] = predict_probs(
    eval_df["comment_text"].tolist(),
    model,
    tokenizer
)
eval_df["clean_pred"] = (eval_df["clean_prob"] >= BEST_THRESHOLD).astype(int)

print(eval_df[["comment_text", "label", "clean_prob", "clean_pred"]].head())

                                        comment_text  label  clean_prob  \
0  Your point is well taken Danno!\n\nIt's intere...      0    0.004654   
1  But if the only witness is the victim, what then?      0    0.006027   
2  Fascinating work, not so much the attack on Br...      0    0.005367   
3  All taxes, except those that you do not pay, a...      0    0.002821   
4  Sorry to say Mr. Brule it is Obama who has pla...      1    0.095400   

   clean_pred  
0           0  
1           0  
2           0  
3           0  
4           0  


Build perturbation function

In [8]:
ZERO_WIDTH = "\u200b"

HOMOGLYPHS = {
    "a": "а",   # Cyrillic a
    "e": "е",   # Cyrillic e
    "o": "о",   # Cyrillic o
    "c": "с",
    "p": "р",
    "x": "х",
    "y": "у",
    "i": "і"
}

TOXIC_LOOKING_WORDS = {
    "hate", "idiot", "stupid", "kill", "ugly", "trash", "moron",
    "racist", "dumb", "die", "loser", "terrorist", "scum",
    "bitch", "fuck", "fucking", "nigger", "retard", "asshole"
}

def add_zero_width(word):
    chars = list(word)
    out = []
    for i, ch in enumerate(chars):
        out.append(ch)
        if i % 2 == 1:
            out.append(ZERO_WIDTH)
    return "".join(out)

def replace_homoglyphs(word):
    out = []
    for ch in word:
        if ch.lower() in HOMOGLYPHS and random.random() < 0.3:
            new_ch = HOMOGLYPHS[ch.lower()]
            if ch.isupper():
                out.append(new_ch.upper())
            else:
                out.append(new_ch)
        else:
            out.append(ch)
    return "".join(out)

def duplicate_random_chars(word, prob=0.2):
    out = []
    for ch in word:
        out.append(ch)
        if random.random() < prob:
            out.append(ch)
    return "".join(out)

def perturb(text):
    words = text.split()
    new_words = []

    for w in words:
        base = re.sub(r"[^a-zA-Z]", "", w).lower()
        new_w = w

        if base in TOXIC_LOOKING_WORDS:
            new_w = add_zero_width(new_w)

        new_w = replace_homoglyphs(new_w)
        new_w = duplicate_random_chars(new_w, prob=0.2)

        new_words.append(new_w)

    return " ".join(new_words)

Select attack sample for Attack 1

In [9]:
attack_candidates = eval_df[
    (eval_df["clean_pred"] == 1) &
    (eval_df["clean_prob"] >= 0.7)
].copy()

print("Eligible toxic high-confidence comments:", len(attack_candidates))

attack_sample = attack_candidates.sample(
    n=min(500, len(attack_candidates)),
    random_state=SEED
).copy()

print("Attack sample size:", len(attack_sample))

Eligible toxic high-confidence comments: 102
Attack sample size: 102


Apply Attack 1 and score it

In [10]:
attack_sample["perturbed_text"] = attack_sample["comment_text"].apply(perturb)

attack_sample["perturbed_prob"] = predict_probs(
    attack_sample["perturbed_text"].tolist(),
    model,
    tokenizer
)

attack_sample["perturbed_pred"] = (
    attack_sample["perturbed_prob"] >= BEST_THRESHOLD
).astype(int)

attack_success_rate = (attack_sample["perturbed_pred"] == 0).mean()
avg_conf_before = attack_sample["clean_prob"].mean()
avg_conf_after = attack_sample["perturbed_prob"].mean()

print("Attack Success Rate (ASR):", attack_success_rate)
print("Average confidence before attack:", avg_conf_before)
print("Average confidence after attack :", avg_conf_after)

Attack Success Rate (ASR): 0.9215686274509803
Average confidence before attack: 0.9238918
Average confidence after attack : 0.08216429


Show a few examples of perturbation

In [11]:
attack_sample[[
    "comment_text",
    "perturbed_text",
    "clean_prob",
    "perturbed_prob",
    "clean_pred",
    "perturbed_pred"
]].head(10)

,comment_text,perturbed_text,clean_prob,perturbed_prob,clean_pred,perturbed_pred
741,"I'm seeing in the comments a general "" what th...",II'm sseeeingg іn thее ccommmeents a generaal ...,0.927583,0.007213,1,0
1404,Leah would not get onto an airplane unless it ...,Leah wouldd nоt ggеt oonto аn airplаnnе unlles...,0.970096,0.007824,1,0
1331,Remember those days?? That was Alaska!! With t...,Reememberr those daуs?? Thhaat was Allasska!!!...,0.973290,0.010683,1,0
1105,"Well! This Virginia guy must be a like, a Xeno...",Well!! TThiiss VVіrgііnniа guy mmustt be a lіk...,0.882615,0.013108,1,0
1011,"Now the AG, the governor, and the rest of us l...","Nоw tthe АG, thе ggovernnоr, aand tthe rrеst o...",0.977435,0.008034,1,0
981,"Sell that crap elsewhere, Boris...","Selll thаt craap ellsewherrе, Boriss...",0.969677,0.007397,1,0
1848,132 tons of trash.The bums have nothing to do ...,1322 tons of trassh..The bumms havee nnoothing...,0.962686,0.009365,1,0
1071,You don't know what she was going threw. Look ...,Уоuu don''t knoow wwhаt shе wwas gоіng tthrew....,0.842985,0.069431,1,0
317,Nikki Haley is a disgrace to the international...,NNіkki Haley іs a disggrace tto thе intееrnaat...,0.812918,0.009069,1,0
48,"Nicky, you can't handle the truth. Both Harper...","NNiicckky, уou cаn't hааnnddle thee trruth. BB...",0.901038,0.010124,1,0


ASR summary table

In [12]:
attack1_results = pd.DataFrame({
    "Metric": [
        "Sample Size",
        "Attack Success Rate",
        "Average Confidence Before",
        "Average Confidence After"
    ],
    "Value": [
        len(attack_sample),
        attack_success_rate,
        avg_conf_before,
        avg_conf_after
    ]
})

attack1_results

,Metric,Value
0,Sample Size,102.000000
1,Attack Success Rate,0.921569
2,Average Confidence Before,0.923892
3,Average Confidence After,0.082164


Create poisoned training set for Attack 2

In [13]:
poisoned_train_df = train_df.copy()

flip_n = int(0.05 * len(poisoned_train_df))
flip_indices = np.random.choice(poisoned_train_df.index, size=flip_n, replace=False)

poisoned_train_df.loc[flip_indices, "label"] = 1 - poisoned_train_df.loc[flip_indices, "label"]

print("Training rows:", len(poisoned_train_df))
print("Flipped rows:", flip_n)

Training rows: 10000
Flipped rows: 500


Prepare poisoned datasets for retraining

In [14]:
poison_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_batch(batch):
    return poison_tokenizer(
        batch["comment_text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN
    )

poison_train_hf = Dataset.from_pandas(
    poisoned_train_df[["comment_text", "label"]],
    preserve_index=False
)

poison_eval_hf = Dataset.from_pandas(
    eval_df[["comment_text", "label"]],
    preserve_index=False
)

poison_train_hf = poison_train_hf.map(tokenize_batch, batched=True)
poison_eval_hf = poison_eval_hf.map(tokenize_batch, batched=True)

poison_train_hf = poison_train_hf.remove_columns(["comment_text"])
poison_eval_hf = poison_eval_hf.remove_columns(["comment_text"])

poison_train_hf.set_format("torch")
poison_eval_hf.set_format("torch")

print(poison_train_hf)
print(poison_eval_hf)

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Dataset({
    features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 10000
})
Dataset({
    features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 2000
})


Metrics function for poisoned retraining

In [15]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.softmax(torch.tensor(logits), dim=1)[:, 1].numpy()
    preds = (probs >= BEST_THRESHOLD).astype(int)

    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro")
    }

Load fresh DistilBERT for poisoned retraining

In [16]:
poison_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Training args for poisoned retraining

In [17]:
poison_training_args = TrainingArguments(
    output_dir=POISONED_MODEL_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=50,
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    num_train_epochs=1,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    report_to="none",
    seed=SEED
)

print(poison_training_args)

TrainingArguments(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
enable_jit_checkpoint=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=epoch,
eval_use_gather_object=False,
fp16=False,
fp

Create poisoned trainer

In [18]:
poison_trainer = Trainer(
    model=poison_model,
    args=poison_training_args,
    train_dataset=poison_train_hf,
    eval_dataset=poison_eval_hf,
    compute_metrics=compute_metrics
)

print("Poison trainer created.")

Poison trainer created.


Train poisoned model

In [19]:
print("Training poisoned model...")
poison_trainer.train()

Training poisoned model...


f:\areeba responsible\.venv\lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.393185,0.176402,0.940000,0.786398


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=1250, training_loss=0.3453445205688477, metrics={'train_runtime': 2134.2235, 'train_samples_per_second': 4.686, 'train_steps_per_second': 0.586, 'total_flos': 331168496640000.0, 'train_loss': 0.3453445205688477, 'epoch': 1.0})

Evaluate poisoned model

In [20]:
poison_pred_output = poison_trainer.predict(poison_eval_hf)
poison_logits = poison_pred_output.predictions
poison_y_true = poison_pred_output.label_ids
poison_y_prob = torch.softmax(torch.tensor(poison_logits), dim=1)[:, 1].numpy()
poison_y_pred = (poison_y_prob >= BEST_THRESHOLD).astype(int)

print("Poisoned evaluation complete.")

f:\areeba responsible\.venv\lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Poisoned evaluation complete.


Compare baseline vs poisoned

In [21]:
baseline_y_true = eval_df["label"].values
baseline_y_pred = eval_df["clean_pred"].values

baseline_cm = confusion_matrix(baseline_y_true, baseline_y_pred, labels=[0, 1])
b_tn, b_fp, b_fn, b_tp = baseline_cm.ravel()
baseline_fnr = b_fn / (b_fn + b_tp) if (b_fn + b_tp) > 0 else 0.0

poison_cm = confusion_matrix(poison_y_true, poison_y_pred, labels=[0, 1])
p_tn, p_fp, p_fn, p_tp = poison_cm.ravel()
poison_fnr = p_fn / (p_fn + p_tp) if (p_fn + p_tp) > 0 else 0.0

comparison_table = pd.DataFrame({
    "Metric": ["Accuracy", "F1 Macro", "False Negative Rate"],
    "Baseline": [
        accuracy_score(baseline_y_true, baseline_y_pred),
        f1_score(baseline_y_true, baseline_y_pred, average="macro"),
        baseline_fnr
    ],
    "Poisoned": [
        accuracy_score(poison_y_true, poison_y_pred),
        f1_score(poison_y_true, poison_y_pred, average="macro"),
        poison_fnr
    ]
})

comparison_table

,Metric,Baseline,Poisoned
0,Accuracy,0.942500,0.940000
1,F1 Macro,0.788243,0.786398
2,False Negative Rate,0.443750,0.425000


Part 3 outputs

In [22]:
attack1_results.to_csv("saved_models/part3_attack1_results.csv", index=False)
comparison_table.to_csv("saved_models/part3_attack2_comparison.csv", index=False)

poison_trainer.save_model(POISONED_MODEL_DIR)
poison_tokenizer.save_pretrained(POISONED_MODEL_DIR)

print("Saved:")
print(" - saved_models/part3_attack1_results.csv")
print(" - saved_models/part3_attack2_comparison.csv")
print(" -", POISONED_MODEL_DIR)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved:
 - saved_models/part3_attack1_results.csv
 - saved_models/part3_attack2_comparison.csv
 - saved_models/part3_poisoned_model
